# Análisis comparativo de correlaciones por modelo

Exploramos las correlaciones entre las métricas featurizadas y el resultado binario `WL_NUM` para comparar la señal que aporta cada versión del pipeline: **Baseline**, **Venue+** y **Enhanced**. Replicamos el análisis clásico de correlaciones, pero lo aplicamos de forma independiente a los tres conjuntos de features que se utilizan en `02_WL_estimator.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display


import importlib.util, sys
spec = importlib.util.spec_from_file_location("feature_functions", "02_FeatureFunctions.py")
feature_functions = importlib.util.module_from_spec(spec)
sys.modules["feature_functions"] = feature_functions
spec.loader.exec_module(feature_functions)
from feature_functions import build_match_dataset_enhanced

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


In [ ]:
# === Configuración ===
BASE_DIR = Path("/Users/pablo/Documents/BigData/BasketballAnalysis")
DATA_ROOT = BASE_DIR / "00_data/00d_featurized/2024-25"
FEATURIZED_PATH = DATA_ROOT / "teamgamelogs_featurized.parquet"

TARGET_COLUMN = "WL_NUM"
MAX_FEATURES_HEATMAP = 20
TOP_K_SCATTER = 5

if not FEATURIZED_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset featurizado en {FEATURIZED_PATH}.")
else:
    print(f"✅ Dataset encontrado en {FEATURIZED_PATH}")


In [ ]:
# === Carga de datos ===
df = pd.read_parquet(FEATURIZED_PATH)
if TARGET_COLUMN not in df.columns:
    raise KeyError(f"La columna objetivo '{TARGET_COLUMN}' no está presente en el dataset featurizado.")

print(f"Shape del dataset: {df.shape}")


In [ ]:
# === Dataset a nivel partido (diferenciales) ===
ADVANCED_FEATURES = [
    'WIN_STREAK', 'LAST_5_PCT', 'DAYS_REST', 'SEASON_W_PCT',
    'PACE', 'TURNOVER_RATIO'
]

X_all, y_all, meta_all = build_match_dataset_enhanced(
    df,
    numeric_feature_prefixes=("ROLL10_", "VENUE_"),
    advanced_features=ADVANCED_FEATURES,
)

if 'DIFF_B2B_FLAG' in X_all.columns:
    X_all = X_all.rename(columns={'DIFF_B2B_FLAG': 'B2B_ADVANTAGE'})

match_df = X_all.copy()
match_df[TARGET_COLUMN] = y_all

print(f"Shape dataset partido: {match_df.shape}")
match_df.head()


## Conjuntos de features del pipeline

Tomamos exactamente las mismas listas de features que se utilizan en `02_WL_estimator.ipynb`. Cada conjunto representa una iteración del pipeline, añadiendo nuevas señales sobre la versión anterior.

In [ ]:
# === Definición de features por modelo (copiado de 02_WL_estimator.ipynb) ===

def unique_preserve_order(seq):
    seen = set()
    ordered = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            ordered.append(item)
    return ordered

baseline_features = [c for c in match_df.columns if c.startswith('DIFF_ROLL10_') or c == 'HOME_COURT']
venue_features = baseline_features + [c for c in ['DIFF_VENUE_W_PCT'] if c in match_df.columns]
enhanced_candidates = [
    'DIFF_WIN_STREAK', 'DIFF_LAST_5_PCT', 'DIFF_DAYS_REST',
    'DIFF_SEASON_W_PCT', 'DIFF_PACE', 'DIFF_TURNOVER_RATIO', 'B2B_ADVANTAGE'
]
enhanced_features = venue_features + [c for c in enhanced_candidates if c in match_df.columns]

baseline_features = unique_preserve_order(baseline_features)
venue_features = unique_preserve_order(venue_features)
enhanced_features = unique_preserve_order(enhanced_features)

all_feature_columns = unique_preserve_order(baseline_features + venue_features + enhanced_features + [TARGET_COLUMN])
numeric_df = match_df[all_feature_columns].apply(pd.to_numeric, errors='coerce')
valid_numeric_columns = set(numeric_df.columns)
valid_numeric_columns.discard(TARGET_COLUMN)

feature_sets = {
    'Baseline': [f for f in baseline_features if f in valid_numeric_columns],
    'Venue+': [f for f in venue_features if f in valid_numeric_columns],
    'Enhanced': [f for f in enhanced_features if f in valid_numeric_columns],
}

MODEL_ORDER = list(feature_sets.keys())
incremental_feature_sets = {}
for idx, name in enumerate(MODEL_ORDER):
    if idx == 0:
        incremental_feature_sets[name] = feature_sets[name]
    else:
        prev = set(feature_sets[MODEL_ORDER[idx - 1]])
        incremental_feature_sets[name] = [f for f in feature_sets[name] if f not in prev]

features_overview = (
    pd.DataFrame(
        [(name, len(cols)) for name, cols in feature_sets.items()],
        columns=['Modelo', 'Nº de features']
    )
    .set_index('Modelo')
)

new_features_overview = (
    pd.DataFrame(
        [(name, len(cols)) for name, cols in incremental_feature_sets.items()],
        columns=['Modelo', 'Nº features nuevas vs anterior']
    )
    .set_index('Modelo')
)

display(features_overview.join(new_features_overview))
for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")
    print(', '.join(cols))
    if incremental_feature_sets[name] and name != MODEL_ORDER[0]:
        print(f"  Nuevas vs anterior ({len(incremental_feature_sets[name])}): {', '.join(incremental_feature_sets[name])}")
    elif name == MODEL_ORDER[0]:
        print("  Nuevas vs anterior: modelo inicial")
    else:
        print("  Nuevas vs anterior: --")
    print()


### Qué aporta cada versión del modelo

- **Baseline**: incluye las diferencias en métricas `ROLL10` entre equipos y la bandera `HOME_COURT`, capturando la forma reciente y la ventaja de localía.
- **Venue+**: añade el rendimiento relativo por tipo de venue (`DIFF_VENUE_W_PCT`), lo que introduce información contextual sobre canchas neutrales o visitas prolongadas.
- **Enhanced**: suma rachas (`DIFF_WIN_STREAK`), estado reciente (`DIFF_LAST_5_PCT`, `DIFF_SEASON_W_PCT`), descanso (`DIFF_DAYS_REST`, `B2B_ADVANTAGE`) y ajustes de ritmo/seguridad del balón (`DIFF_PACE`, `DIFF_TURNOVER_RATIO`).

En los análisis siguientes distinguimos entre las señales heredadas y las nuevas de cada iteración para resaltar qué aporta cada bloque adicional.


## Funciones auxiliares para el análisis

Utilizamos utilidades comunes para replicar el análisis de correlaciones en cada conjunto de features.

In [ ]:
def compute_correlations(frame, features, target):
    available = [f for f in features if f in frame.columns]
    if not available:
        return pd.Series(dtype=float)
    corr = frame[available + [target]].corr()[target].drop(target)
    return corr.dropna().sort_values(ascending=False)


def summarize_correlations(corr, top_n_pos=10, top_n_neg=10, weakest_n=5):
    if corr.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float), pd.Series(dtype=float)
    top_pos = corr.sort_values(ascending=False).head(min(top_n_pos, len(corr)))
    top_neg = corr.sort_values().head(min(top_n_neg, len(corr)))
    weakest = corr.abs().sort_values().head(min(weakest_n, len(corr)))
    return top_pos, top_neg, weakest


def plot_heatmap(frame, features, target, title, max_features=20):
    if not features:
        print(f"No hay features disponibles para {title}.")
        return
    ordered = frame[features + [target]].corr()[target].abs().sort_values(ascending=False).index.tolist()
    top_cols = [col for col in ordered if col != target][:max_features]
    corr_matrix = frame[top_cols + [target]].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
    plt.title(title)
    plt.tight_layout()


def plot_scatter_grid(frame, features, target, title, top_k=5):
    if not features:
        print(f"No hay features disponibles para {title}.")
        return
    corr = frame[features + [target]].corr()[target].drop(target).abs().sort_values(ascending=False)
    top_cols = corr.head(min(top_k, len(corr))).index.tolist()
    n_cols = len(top_cols)
    fig, axes = plt.subplots(nrows=1, ncols=n_cols, figsize=(4 * n_cols, 4), sharey=True)
    if n_cols == 1:
        axes = [axes]
    for ax, col in zip(axes, top_cols):
        sns.scatterplot(data=frame, x=col, y=target, alpha=0.3, ax=ax)
        ax.set_title(col)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()


## Resumen de correlaciones por modelo

Calculamos las correlaciones con `WL_NUM` para cada conjunto y listamos las señales más fuertes, las más débiles y aquellas con correlación negativa más marcada.

In [ ]:
analysis_state = {
    'model_correlations': {},
    'model_rankings': {},
    'incremental_correlations': {},
    'incremental_rankings': {},
}

for idx, model_name in enumerate(MODEL_ORDER):
    print(f"=== {model_name} ===")
    features = feature_sets[model_name]
    corr = compute_correlations(numeric_df, features, TARGET_COLUMN)
    analysis_state['model_correlations'][model_name] = corr

    top_pos, top_neg, weakest = summarize_correlations(corr)
    analysis_state['model_rankings'][model_name] = {
        'top_pos': top_pos,
        'top_neg': top_neg,
        'weakest': weakest,
    }

    print('Top 10 correlaciones positivas (todas las features disponibles)')
    display(top_pos.to_frame(name='Correlación'))

    print('Top 10 correlaciones negativas (todas las features disponibles)')
    display(top_neg.to_frame(name='Correlación'))

    print('5 correlaciones más débiles (en valor absoluto)')
    display(weakest.to_frame(name='|Correlación|'))

    new_features = incremental_feature_sets[model_name]
    available_new = [f for f in new_features if f in corr.index]
    new_corr = corr.loc[available_new] if available_new else pd.Series(dtype=float)
    analysis_state['incremental_correlations'][model_name] = new_corr

    if idx > 0:
        print('---')
        print(f"Señales nuevas introducidas en {model_name}")
        if new_corr.empty:
            print('No hay correlaciones disponibles para las nuevas features.')
            analysis_state['incremental_rankings'][model_name] = {
                'top_pos': pd.Series(dtype=float),
                'top_neg': pd.Series(dtype=float),
                'weakest': pd.Series(dtype=float),
            }
        else:
            new_top_pos, new_top_neg, new_weakest = summarize_correlations(new_corr)
            analysis_state['incremental_rankings'][model_name] = {
                'top_pos': new_top_pos,
                'top_neg': new_top_neg,
                'weakest': new_weakest,
            }
            print('Top correlaciones positivas (features nuevas)')
            display(new_top_pos.to_frame(name='Correlación'))
            print('Top correlaciones negativas (features nuevas)')
            display(new_top_neg.to_frame(name='Correlación'))
            print('5 correlaciones más débiles (features nuevas)')
            display(new_weakest.to_frame(name='|Correlación|'))
    else:
        analysis_state['incremental_rankings'][model_name] = {
            'top_pos': top_pos,
            'top_neg': top_neg,
            'weakest': weakest,
        }

    print()

model_correlations = analysis_state['model_correlations']
model_rankings = analysis_state['model_rankings']
incremental_correlations = analysis_state['incremental_correlations']
incremental_rankings = analysis_state['incremental_rankings']

globals().update({
    'model_correlations': model_correlations,
    'model_rankings': model_rankings,
    'incremental_correlations': incremental_correlations,
    'incremental_rankings': incremental_rankings,
})


## Heatmaps de las features más correlacionadas

Visualizamos la matriz de correlaciones entre las features más influyentes de cada modelo y el objetivo, replicando el análisis original para cada conjunto.

In [ ]:
for model_name, corr in model_correlations.items():
    title = f"Correlaciones (top {MAX_FEATURES_HEATMAP} features vs WL_NUM) - {model_name}"
    plot_heatmap(numeric_df, feature_sets[model_name], TARGET_COLUMN, title, max_features=MAX_FEATURES_HEATMAP)


## Dispersión de las correlaciones más fuertes

Mostramos la relación entre `WL_NUM` y las cinco features con mayor correlación absoluta dentro de cada modelo.

In [ ]:
for model_name in feature_sets:
    title = f"Dispersión vs WL_NUM (top {TOP_K_SCATTER} correlaciones absolutas) - {model_name}"
    plot_scatter_grid(numeric_df, feature_sets[model_name], TARGET_COLUMN, title, top_k=TOP_K_SCATTER)


## Comparativa entre modelos

Evaluamos cómo evoluciona la fuerza de las correlaciones al pasar del modelo Baseline al Venue+ y al Enhanced.

In [ ]:
# Evolución de la correlación máxima (positiva, absoluta y nuevas features)
summary_rows = []
for model_name in MODEL_ORDER:
    corr = model_correlations.get(model_name, pd.Series(dtype=float))
    if corr.empty:
        continue
    max_pos_feature = corr.idxmax()
    max_pos_value = corr.loc[max_pos_feature]
    max_abs_feature = corr.abs().idxmax()
    max_abs_value = corr.abs().loc[max_abs_feature]

    new_corr = incremental_correlations.get(model_name, pd.Series(dtype=float))
    if not new_corr.empty:
        new_abs = new_corr.abs()
        top_new_feature = new_abs.idxmax()
        top_new_value = new_corr.loc[top_new_feature]
    else:
        top_new_feature = '—'
        top_new_value = np.nan

    summary_rows.append({
        'Modelo': model_name,
        'Feature correlación positiva máxima': max_pos_feature,
        'Correlación positiva máxima': max_pos_value,
        'Feature correlación absoluta máxima': max_abs_feature,
        'Correlación absoluta máxima': max_abs_value,
        'Feature nueva más correlacionada': top_new_feature,
        'Correlación feature nueva': top_new_value,
    })

max_corr_df = pd.DataFrame(summary_rows).set_index('Modelo')
display(max_corr_df)

plt.figure(figsize=(8, 4))
plt.plot(max_corr_df.index, max_corr_df['Correlación positiva máxima'], marker='o', label='Correlación positiva máxima')
plt.plot(max_corr_df.index, max_corr_df['Correlación absoluta máxima'], marker='o', label='Correlación absoluta máxima')
if not max_corr_df['Correlación feature nueva'].isna().all():
    plt.plot(max_corr_df.index, max_corr_df['Correlación feature nueva'], marker='o', label='Correlación feature nueva')
plt.title('Evolución de las correlaciones máximas por modelo')
plt.ylabel('Coeficiente de correlación')
plt.ylim(-1, 1)
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()


In [ ]:
# Tabla comparativa de poder predictivo (medimos estadísticos sobre las correlaciones absolutas)
power_rows = []
for model_name in MODEL_ORDER:
    corr = model_correlations.get(model_name, pd.Series(dtype=float))
    if corr.empty:
        continue
    abs_corr = corr.abs().sort_values(ascending=False)
    row = {
        'Modelo': model_name,
        'Media |corr| top 5': abs_corr.head(5).mean(),
        'Media |corr| top 10': abs_corr.head(10).mean(),
        'Mediana |corr| top 10': abs_corr.head(10).median(),
        'Máx |corr|': abs_corr.iloc[0],
        'Mín |corr| top 10': abs_corr.head(10).iloc[-1] if len(abs_corr) >= 10 else abs_corr.iloc[-1],
    }

    new_corr = incremental_correlations.get(model_name, pd.Series(dtype=float))
    if not new_corr.empty:
        abs_new_corr = new_corr.abs().sort_values(ascending=False)
        row.update({
            'Media |corr| (features nuevas)': abs_new_corr.mean(),
            'Máx |corr| (features nuevas)': abs_new_corr.iloc[0],
        })
    else:
        row.update({
            'Media |corr| (features nuevas)': np.nan,
            'Máx |corr| (features nuevas)': np.nan,
        })

    power_rows.append(row)

power_df = pd.DataFrame(power_rows).set_index('Modelo')
display(power_df)


### Heatmaps comparativos

Colocamos los heatmaps de los tres modelos en paralelo para visualizar rápidamente las diferencias en bloques de correlación y, posteriormente, nos enfocamos solo en las señales nuevas de Venue+ y Enhanced.


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(MODEL_ORDER), figsize=(5 * len(MODEL_ORDER), 4), constrained_layout=True)
if len(MODEL_ORDER) == 1:
    axes = [axes]
for ax, model_name in zip(axes, MODEL_ORDER):
    features = feature_sets[model_name]
    corr = model_correlations.get(model_name, pd.Series(dtype=float))
    if corr.empty:
        ax.set_visible(False)
        continue
    ordered = corr.abs().sort_values(ascending=False).index.tolist()
    top_cols = ordered[:MAX_FEATURES_HEATMAP]
    corr_matrix = numeric_df[top_cols + [TARGET_COLUMN]].corr()
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, ax=ax)
    ax.set_title(model_name)
plt.suptitle('Heatmaps comparativos (top features por modelo)', y=1.03)


### Heatmaps de las nuevas features

Resaltamos únicamente las columnas añadidas en cada salto del pipeline para aislar su contribución específica.


In [ ]:
models_with_new = [name for idx, name in enumerate(MODEL_ORDER) if idx > 0 and incremental_feature_sets[name]]
if not models_with_new:
    print('No hay nuevas features que visualizar en heatmaps separados.')
else:
    fig, axes = plt.subplots(nrows=1, ncols=len(models_with_new), figsize=(5 * len(models_with_new), 4), constrained_layout=True)
    if len(models_with_new) == 1:
        axes = [axes]
    for ax, model_name in zip(axes, models_with_new):
        new_features = incremental_feature_sets[model_name]
        corr = incremental_correlations.get(model_name, pd.Series(dtype=float))
        if corr.empty:
            ax.set_visible(False)
            continue
        ordered = corr.abs().sort_values(ascending=False).index.tolist()
        top_cols = ordered[:MAX_FEATURES_HEATMAP]
        subset = numeric_df[top_cols + [TARGET_COLUMN]]
        corr_matrix = subset.corr()
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, ax=ax)
        ax.set_title(f'{model_name} (nuevas)')
    plt.suptitle('Heatmaps de nuevas features por modelo', y=1.03)


## Qué nuevas features aportan más valor

Identificamos qué columnas incorporadas en cada versión posterior ofrecen las correlaciones absolutas más altas con respecto a `WL_NUM`.

In [ ]:
def display_new_feature_summary(model_name, previous_model=None):
    new_cols = incremental_feature_sets[model_name]
    if previous_model is None:
        header = f"Impacto de las features del modelo {model_name} (modelo inicial)"
    else:
        header = f"Impacto de las nuevas features del modelo {model_name} (vs {previous_model})"
    print(header)
    if not new_cols:
        print('No se añadieron nuevas features en esta versión.')
        print()
        return

    corr = model_correlations.get(model_name, pd.Series(dtype=float))
    available = [c for c in new_cols if c in corr.index]
    if not available:
        print('No hay correlaciones disponibles para estas columnas.')
        print()
        return

    series = corr.loc[available]
    ordered = series.abs().sort_values(ascending=False)
    table = pd.DataFrame({
        'Correlación': series.loc[ordered.index],
        '|Correlación|': ordered,
    })
    max_rows = min(20, len(table))
    display(table.head(max_rows))
    if len(table) > max_rows:
        print(f'Se muestran los {max_rows} valores más altos de {len(table)} columnas disponibles.')

    top_pos, top_neg, weakest = summarize_correlations(series, top_n_pos=5, top_n_neg=5, weakest_n=min(5, len(series)))
    if not top_pos.empty:
        print('Top correlaciones positivas (features nuevas)')
        display(top_pos.to_frame(name='Correlación'))
    if not top_neg.empty:
        print('Top correlaciones negativas (features nuevas)')
        display(top_neg.to_frame(name='Correlación'))
    if not weakest.empty:
        print('Correlaciones más débiles (features nuevas)')
        display(weakest.to_frame(name='|Correlación|'))
    print()

for idx, model_name in enumerate(MODEL_ORDER):
    previous = MODEL_ORDER[idx - 1] if idx > 0 else None
    display_new_feature_summary(model_name, previous_model=previous)


### Interpretación de las mejoras

- **Del Baseline al Venue+**: el rendimiento relativo por tipo de venue permite capturar diferencias que las medias móviles no recogen, destacando qué equipos mejoran o empeoran fuera de casa y mostrando su correlación específica sin quedar eclipsada por las señales heredadas.
- **Del Venue+ al Enhanced**: las señales de rachas, descanso y estado reciente refuerzan la capacidad predictiva al reflejar la forma actual del equipo y su desgaste, aportando correlaciones absolutas más altas dentro del bloque de nuevas features.
- **Features más valiosas**: observamos qué variables nuevas (por ejemplo, `DIFF_WIN_STREAK` o `DIFF_DAYS_REST`) aparecen entre las correlaciones más fuertes de sus respectivos incrementos, guiando prioridades de ingeniería de features.
